# Cross-species MI overlap on the all-cells arms
Arms (results/mi_dpt/, rules in `workflow/rules/11_mi_pseudotime.smk`): `pt_harmonized_mouse_nall` (pt_harmonized_rank, 42,290 nuclei), `pt_harmonized_human_nall` (pt_harmonized_rank, 47,533 nuclei), controls `pt_harmonized_unind_mouse_nall` / `pt_harmonized_indmatch_mouse_nall` (within-sample rank of pt_harmonized, 9,333 nuclei each). Every arm scores all of its nuclei (no estimator subsample) on genes detected in at least 1% of them, with 1,001 permutations. Source objects: /n/netscratch/eddy_lab/Everyone/nhilgert/2026_10_22_flagged_harmonized/. Output dir `results/mi_dpt/overlap_harmonized_nall/`. Same analysis as `mi_dpt_overlap_harmonized.ipynb` (3,000-nucleus estimator subsample arms), with only the arm names and output directory changed.


# dpt_pseudotime MI — cross-species overlap

**What ran.** Four MI estimations of `MI(log1p(CP10K) expression ; quantile-normalised dpt_pseudotime)`,
each 200 gene-chunks × 1001 permutations on every nucleus of the arm, genes detected in ≥ 1% of its nuclei.

| run | source object | cells | pseudotime normalisation |
|---|---|---|---|
| `pt_harmonized_unind_mouse_nall` | netscratch mouse fib/dec | 9,333 (`timepoint == 'uninduced'`) | **rank/N within each of the 2 samples** |
| `pt_harmonized_indmatch_mouse_nall` | netscratch mouse fib/dec | 9,333 (induced, size-matched draw, seed 0) | rank/N within sample |
| `pt_harmonized_mouse_nall` | netscratch mouse fib/dec | 42,290 (decidual_apoptotic excluded) | **rank/N global** |
| `pt_harmonized_human_nall` | netscratch human fib/dec | 47,533 (all) | rank/N global |

**Two things to hold in mind while reading.**

1. **MI is invariant under a monotone transform of either variable**, so rank-normalising cannot create or
   destroy information — it only changes the kNN estimator's local distances. It is applied here for
   *comparability*: raw uninduced dpt spans only 0–0.425 of the global axis, so without per-arm ranking the
   arms would be estimated on coordinates with very different dynamic range.
2. **Global normalisation folds sample identity into the coordinate.** `pt_harmonized_mouse_nall` and `pt_harmonized_human_nall` rank across the whole
   multi-sample object, so those two runs are *pooled-coordinate* estimates, not batch-conditioned ones —
   the opposite of the within-sample conditioning settled on for the isodepth MI. `pt_harmonized_unind_mouse_nall` and
   `pt_harmonized_indmatch_mouse_nall` *are* within-sample. Don't read a pooled number as a conditioned one.

**Ortholog universe: Ensembl 116, strict one-to-one only.** No OMA, no orthogroup/component grouping —
`homology_type == 'ortholog_one2one'`, pivoted on `component_id` to pair `gene_symbol` across species.
That table is long-format and also carries `human_copy_number` / `mouse_copy_number`; a column-name
heuristic silently pairs the copy numbers and yields plausible nonsense, so the pivot is explicit and
asserted below.

In [ ]:
import json, pathlib, itertools, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.stats import hypergeom, spearmanr

X    = pathlib.Path("/n/lab_storage/eddy_lab/users/nhilgert/xmens_final")
MID  = X/"results/mi_dpt"
ORTH = pathlib.Path("/n/lab_storage/eddy_lab/users/nhilgert/human_ivd_repro/orthogroups/"
                    "ensembl_human_mouse_components.tsv")
OUT  = MID/"overlap_harmonized_nall"; OUT.mkdir(parents=True, exist_ok=True)
RUNS = ["pt_harmonized_unind_mouse_nall", "pt_harmonized_indmatch_mouse_nall", "pt_harmonized_mouse_nall", "pt_harmonized_human_nall"]
SPECIES = {"pt_harmonized_unind_mouse_nall":"mouse","pt_harmonized_indmatch_mouse_nall":"mouse","pt_harmonized_mouse_nall":"mouse","pt_harmonized_human_nall":"human"}
TOPFRAC, SEED = 0.10, 0

CHECKS = []
def verify(label, ok, got=None, note=""):
    CHECKS.append((label, bool(ok), got, note))
    print(("PASS  " if ok else "FAIL  ") + label + (f"   [{got}]" if got is not None else "")
          + (f"  {note}" if note else ""))
print("output dir:", OUT)

## 1. Load the MI tables

Each run's `mi_all.csv` plus its build metadata and prep manifest. Runs that haven't finished are reported and skipped rather than silently absent.

In [ ]:
MI, META = {}, {}
for run in RUNS:
    f = MID/run/"mi_all.csv"
    if not f.exists():
        print(f"  {run:20s} NOT FINISHED (no mi_all.csv)"); continue
    d = pd.read_csv(f)
    assert {"gene","mi","p"} <= set(d.columns), f"{run}: unexpected columns {list(d.columns)}"
    d = d.drop_duplicates("gene").set_index("gene")
    MI[run] = d
    bm = MID/run/"build_meta.json"; mf = MID/run/"manifest.json"
    META[run] = dict(build=json.loads(bm.read_text()) if bm.exists() else {},
                     prep=json.loads(mf.read_text()) if mf.exists() else {})
    b, m = META[run]["build"], META[run]["prep"]
    print(f"  {run:20s} {len(d):6,} genes | cells {b.get('n_cells','?'):>7} | norm {b.get('normalization','?'):>13}"
          f" | layer {m.get('layer','?')}")
verify("at least the two headline runs are present",
       {"pt_harmonized_unind_mouse_nall","pt_harmonized_human_nall"} <= set(MI), sorted(MI))
verify("prep derived log1p from counts everywhere",
       all("log1p" in str(META[r]["prep"].get("layer","")) for r in MI),
       {r: META[r]["prep"].get("layer") for r in MI})
verify("uninduced and induced-matched arms have equal n (size-matched)",
       (META.get("pt_harmonized_unind_mouse_nall",{}).get("build",{}).get("n_cells")
        == META.get("pt_harmonized_indmatch_mouse_nall",{}).get("build",{}).get("n_cells"))
       if {"pt_harmonized_unind_mouse_nall","pt_harmonized_indmatch_mouse_nall"} <= set(MI) else None,
       {r: META[r]["build"].get("n_cells") for r in MI})

## 2. Ortholog map — Ensembl 116, one-to-one, pivoted on `component_id`

The assert is the point: it fails loudly if the columns are not what we think, rather than pairing copy numbers.

In [ ]:
O = pd.read_csv(ORTH, sep="\t")
NEED = {"component_id","species","gene_symbol","homology_type","biotype"}
verify("ortholog table schema is as expected", NEED <= set(O.columns), sorted(set(O.columns)))
verify("gene_symbol is not a numeric column (would mean we grabbed copy numbers)",
       not pd.api.types.is_numeric_dtype(O.gene_symbol), str(O.gene_symbol.dtype))

o = O[(O.homology_type == "ortholog_one2one") & (O.biotype == "protein_coding")]
piv = (o.pivot_table(index="component_id", columns="species", values="gene_symbol", aggfunc="first"))
verify("both species columns present after pivot", {"human","mouse"} <= set(piv.columns), sorted(piv.columns))
# a one2one component must contribute exactly one gene per species
sz = o.groupby(["component_id","species"]).size()
verify("every one2one component has exactly 1 gene per species", bool((sz == 1).all()),
       dict(sz.value_counts()))
PAIRS = piv.dropna()[["human","mouse"]].drop_duplicates()
PAIRS = PAIRS[~PAIRS.human.duplicated(keep=False) & ~PAIRS.mouse.duplicated(keep=False)]
verify("1:1 pairs recovered", len(PAIRS) > 12000, f"{len(PAIRS):,} pairs")
print(PAIRS.head(4).to_string(index=False))
print("\nEnsembl release:", sorted(O.source_release.unique()) if "source_release" in O else "n/a")
PAIRS.to_csv(OUT/"ortholog_pairs_one2one.csv", index=False)

## 3. Overlap statistics

For each comparison: restrict to the 1:1 pairs measurable in **both** tables, take each side's top decile by MI, and report the observed overlap against the hypergeometric expectation. Alongside it, a **threshold-free** Spearman correlation of MI ranks — which needs no cutoff and is the statistic to trust if the two disagree.

In [ ]:
def compare(a_run, b_run, topfrac=TOPFRAC):
    """a_run vs b_run over the shared 1:1 ortholog universe."""
    sa, sb = SPECIES[a_run], SPECIES[b_run]
    A, B = MI[a_run], MI[b_run]
    if sa == sb:                       # same species: join on gene symbol directly
        genes = A.index.intersection(B.index)
        ga = gb = pd.Index(genes)
    else:
        p = PAIRS[PAIRS.human.isin(B.index if sb == "human" else A.index)]
        p = p[p.mouse.isin(A.index if sa == "mouse" else B.index)]
        ga = pd.Index(p.mouse if sa == "mouse" else p.human)
        gb = pd.Index(p.human if sb == "human" else p.mouse)
    n = len(ga)
    if n < 100: return None
    mia, mib = A.mi.reindex(ga).to_numpy(), B.mi.reindex(gb).to_numpy()
    ok = np.isfinite(mia) & np.isfinite(mib)
    mia, mib, ga, gb, n = mia[ok], mib[ok], ga[ok], gb[ok], int(ok.sum())
    k = int(round(topfrac * n))
    ta = set(np.argsort(-mia)[:k]); tb = set(np.argsort(-mib)[:k])
    ov = len(ta & tb); exp = k * k / n
    p_hg = float(hypergeom.sf(ov - 1, n, k, k))
    rho = float(spearmanr(mia, mib).statistic)
    return dict(a=a_run, b=b_run, universe=n, k=k, overlap=ov, expected=round(exp, 1),
                fold=round(ov / exp, 3) if exp else np.nan, p=p_hg, rho=round(rho, 4),
                shared_genes_a=[str(ga[i]) for i in sorted(ta & tb)][:40])

COMPARISONS = [("pt_harmonized_unind_mouse_nall","pt_harmonized_human_nall"),      # resting mouse vs human  <- the specificity question
               ("pt_harmonized_indmatch_mouse_nall","pt_harmonized_human_nall"),   # induced, size-matched vs human  <- the control arm
               ("pt_harmonized_mouse_nall","pt_harmonized_human_nall"),        # full cross-species on pseudotime
               ("pt_harmonized_unind_mouse_nall","pt_harmonized_indmatch_mouse_nall"),   # uninduced vs induced, within mouse
               ("pt_harmonized_unind_mouse_nall","pt_harmonized_mouse_nall")]
rows = [r for a, b in COMPARISONS if a in MI and b in MI for r in [compare(a, b)] if r]
R = pd.DataFrame(rows)
print(R[["a","b","universe","k","overlap","expected","fold","p","rho"]].to_string(index=False))
R.drop(columns=["shared_genes_a"]).to_csv(OUT/"overlap_summary.csv", index=False)
verify("every comparison has a positive universe", bool((R.universe > 1000).all()) if len(R) else None,
       dict(zip(R.a + " vs " + R.b, R.universe)) if len(R) else None)

## 4. Is the enrichment more than the pairing? — permutation null

The fold-enrichment above is against a hypergeometric expectation, which assumes the two gene lists are independent draws. A stronger check shuffles the **ortholog pairing** itself and recomputes, so the null preserves each species' MI distribution and only destroys the correspondence.

In [ ]:
def perm_null(a_run, b_run, n_perm=500, topfrac=TOPFRAC, seed=SEED):
    sa, sb = SPECIES[a_run], SPECIES[b_run]
    A, B = MI[a_run], MI[b_run]
    p = PAIRS[PAIRS.human.isin(B.index if sb == "human" else A.index)]
    p = p[p.mouse.isin(A.index if sa == "mouse" else B.index)]
    ga = pd.Index(p.mouse if sa == "mouse" else p.human)
    gb = pd.Index(p.human if sb == "human" else p.mouse)
    mia, mib = A.mi.reindex(ga).to_numpy(), B.mi.reindex(gb).to_numpy()
    ok = np.isfinite(mia) & np.isfinite(mib); mia, mib = mia[ok], mib[ok]
    n = len(mia); k = int(round(topfrac * n))
    ta = set(np.argsort(-mia)[:k])
    obs = len(ta & set(np.argsort(-mib)[:k]))
    rng = np.random.default_rng(seed); null = np.empty(n_perm, int)
    for i in range(n_perm):
        null[i] = len(ta & set(np.argsort(-rng.permutation(mib))[:k]))
    return dict(a=a_run, b=b_run, observed=obs, null_mean=round(float(null.mean()), 1),
                null_sd=round(float(null.std()), 1), null_max=int(null.max()),
                z=round(float((obs - null.mean()) / max(null.std(), 1e-9)), 2),
                p_emp=float((null >= obs).mean()), n_perm=n_perm), null

NULLS = {}
prows = []
for a, b in COMPARISONS[:3]:
    if a in MI and b in MI:
        st, null = perm_null(a, b); NULLS[(a, b)] = null; prows.append(st)
        print(f"{a:20s} vs {b:16s} obs={st['observed']:5d}  null {st['null_mean']}±{st['null_sd']} "
              f"(max {st['null_max']})  z={st['z']:+.1f}  p_emp={st['p_emp']:.3g}")
P = pd.DataFrame(prows)
if len(P): P.to_csv(OUT/"overlap_permutation_null.csv", index=False)
verify("observed overlap exceeds every shuffled-pairing draw",
       bool((P.observed > P.null_max).all()) if len(P) else None,
       dict(zip(P.a, P.observed - P.null_max)) if len(P) else None)

## 5. Cutoff sensitivity

The top-decile threshold is a choice. Sweeping it shows whether the conclusion depends on it — reported whole rather than mined for the best value.

In [ ]:
sw = []
for frac in (0.01, 0.02, 0.05, 0.10, 0.20, 0.30):
    for a, b in COMPARISONS:
        if a in MI and b in MI:
            r = compare(a, b, topfrac=frac)
            if r: sw.append(dict(frac=frac, pair=f"{a} vs {b}", fold=r["fold"],
                                 overlap=r["overlap"], k=r["k"], p=r["p"]))
S = pd.DataFrame(sw)
print(S.pivot_table(index="frac", columns="pair", values="fold").round(3).to_string())
S.to_csv(OUT/"overlap_cutoff_sweep.csv", index=False)
mono = {p: bool(g.sort_values("frac").fold.is_monotonic_decreasing) for p, g in S.groupby("pair")}
# Descriptive, not a requirement -- printed rather than passed to verify(), which would put an
# informational observation into the FAILED tally.
print("fold declines monotonically with a looser cutoff:", mono)

## 6. Figures

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14.5, 4.6))
lab = (R.a.str.replace("dpt_","",regex=False).str.replace("_mouse","",regex=False)
        + "\nvs " + R.b.str.replace("dpt_","",regex=False).str.replace("_human","",regex=False))
ax = axes[0]
ax.bar(range(len(R)), R.fold, color="#4a7fb5", edgecolor="k", linewidth=0.5)
ax.axhline(1.0, c="r", ls="--", lw=1, label="no enrichment")
for i, (f, o, e) in enumerate(zip(R.fold, R.overlap, R.expected)):
    ax.text(i, f + 0.04, f"{f:.2f}\n{o}/{e:.0f}", ha="center", fontsize=8)
ax.set_xticks(range(len(R))); ax.set_xticklabels(lab, fontsize=8)
ax.set_ylabel(f"fold enrichment of shared top-{int(TOPFRAC*100)}% MI genes")
ax.set_ylim(0, max(R.fold.max() * 1.25, 1.4)); ax.legend(fontsize=8, frameon=False)
ax.set_title("Top-decile overlap vs hypergeometric expectation", fontsize=10)

ax = axes[1]
ax.bar(range(len(R)), R.rho, color="#7b9e5b", edgecolor="k", linewidth=0.5)
ax.axhline(0, c="k", lw=0.8)
for i, v in enumerate(R.rho): ax.text(i, v + 0.005, f"{v:+.3f}", ha="center", fontsize=8)
ax.set_xticks(range(len(R))); ax.set_xticklabels(lab, fontsize=8)
ax.set_ylabel("Spearman ρ of MI ranks (threshold-free)")
ax.set_title("Threshold-free agreement over the same universe", fontsize=10)
fig.tight_layout(); fig.savefig(OUT/"overlap_summary.png", dpi=140, bbox_inches="tight")
print("wrote", OUT/"overlap_summary.png")

In [ ]:
pairs_to_plot = [(a, b) for a, b in COMPARISONS[:3] if a in MI and b in MI]
if pairs_to_plot:
    fig, axes = plt.subplots(1, len(pairs_to_plot), figsize=(5.2*len(pairs_to_plot), 5.0), squeeze=False)
    for j, (a, b) in enumerate(pairs_to_plot):
        A, B = MI[a], MI[b]
        p = PAIRS[PAIRS.human.isin(B.index if SPECIES[b]=="human" else A.index)]
        p = p[p.mouse.isin(A.index if SPECIES[a]=="mouse" else B.index)]
        ga = pd.Index(p.mouse if SPECIES[a]=="mouse" else p.human)
        gb = pd.Index(p.human if SPECIES[b]=="human" else p.mouse)
        x, y = A.mi.reindex(ga).to_numpy(), B.mi.reindex(gb).to_numpy()
        ok = np.isfinite(x) & np.isfinite(y); x, y = x[ok], y[ok]
        rx, ry = pd.Series(x).rank(pct=True), pd.Series(y).rank(pct=True)
        ax = axes[0][j]
        both = (rx > 1-TOPFRAC) & (ry > 1-TOPFRAC)
        ax.scatter(rx[~both], ry[~both], s=2.5, c="0.72", rasterized=True)
        ax.scatter(rx[both], ry[both], s=6, c="#c0392b", rasterized=True, label=f"top decile both (n={int(both.sum())})")
        ax.axvline(1-TOPFRAC, c="k", ls=":", lw=0.8); ax.axhline(1-TOPFRAC, c="k", ls=":", lw=0.8)
        ax.set_xlabel(f"{a}\nMI percentile"); ax.set_ylabel(f"{b}\nMI percentile")
        ax.set_title(f"ρ = {spearmanr(x, y).statistic:+.3f}   n = {len(x):,} 1:1 pairs", fontsize=10)
        ax.legend(fontsize=8, loc="upper left", frameon=False); ax.set_aspect("equal")
    fig.suptitle("MI rank agreement over strict 1:1 Ensembl orthologs", y=1.02)
    fig.tight_layout(); fig.savefig(OUT/"mi_rank_scatter.png", dpi=140, bbox_inches="tight")
    print("wrote", OUT/"mi_rank_scatter.png")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.4))
ax = axes[0]
for run in MI:
    v = MI[run].mi.to_numpy(); v = v[np.isfinite(v)]
    ax.plot(np.sort(v), np.linspace(0, 1, len(v)), lw=1.6, label=f"{run} (n={len(v):,})")
ax.set_xlabel("MI (nats)"); ax.set_ylabel("ECDF"); ax.set_xscale("symlog", linthresh=1e-3)
ax.legend(fontsize=7.5, frameon=False); ax.set_title("MI distribution per run", fontsize=10)

ax = axes[1]
if len(S):
    for p, g in S.groupby("pair"):
        g = g.sort_values("frac"); ax.plot(g.frac*100, g.fold, "o-", lw=1.5, ms=4, label=p)
    ax.axhline(1, c="r", ls="--", lw=1); ax.set_xscale("log")
    ax.set_xlabel("top-% cutoff"); ax.set_ylabel("fold enrichment")
    ax.legend(fontsize=7, frameon=False); ax.set_title("Cutoff sensitivity", fontsize=10)

ax = axes[2]
if NULLS:
    (a, b), null = next(iter(NULLS.items()))
    obs = int(P[(P.a == a) & (P.b == b)].observed.iloc[0])
    ax.hist(null, bins=30, color="0.75", edgecolor="k", linewidth=0.4, label="shuffled pairing")
    ax.axvline(obs, c="#c0392b", lw=2, label=f"observed = {obs}")
    ax.set_xlabel("shared top-decile genes"); ax.set_ylabel("permutations")
    ax.legend(fontsize=8, frameon=False)
    ax.set_title(f"Permutation null\n{a} vs {b}", fontsize=10)
fig.tight_layout(); fig.savefig(OUT/"mi_diagnostics.png", dpi=140, bbox_inches="tight")
print("wrote", OUT/"mi_diagnostics.png")

## 7. The shared genes

Top-decile in both arms, for the two comparisons that carry the argument.

In [ ]:
shared = {}
for a, b in COMPARISONS:
    if a in MI and b in MI:
        r = compare(a, b)
        if r: shared[f"{a} vs {b}"] = r["shared_genes_a"]
for k, v in shared.items():
    print(f"\n{k}  (first {len(v)} shown)")
    print("   " + ", ".join(v))
json.dump(shared, open(OUT/"shared_top_decile_genes.json","w"), indent=1)

rep = dict(runs={r: dict(META[r]["build"], **{"prep_layer": META[r]["prep"].get("layer"),
                                              "prep_n_genes": META[r]["prep"].get("n_genes")}) for r in MI},
           ortholog_source=str(ORTH), ortholog_release=sorted(O.source_release.unique()) if "source_release" in O else None,
           n_one2one_pairs=int(len(PAIRS)), topfrac=TOPFRAC,
           overlap=R.drop(columns=["shared_genes_a"]).to_dict("records") if len(R) else [],
           permutation_null=P.to_dict("records") if len(P) else [],
           checks=[dict(label=l, ok=o, got=str(g)) for l, o, g, _ in CHECKS])
json.dump(rep, open(OUT/"overlap_report.json","w"), indent=1, default=str)
bad = [l for l, o, _, _ in CHECKS if not o]
print(f"\n{len(CHECKS)} checks | {len(bad)} FAILED" + (": " + "; ".join(bad) if bad else " — all passed"))
print("outputs in", OUT)

---
# 8. Cross-species clustering of expression trajectories over pseudotime

MI is **unsigned** — a gene with high MI carries information about position on the axis, but MI cannot say
whether it rises or falls, or whether it does the same thing in both species. This section clusters the
*trajectory shapes* directly.

**Construction.** For each conserved gene pair (top decile of MI in **both** full-object runs):

1. bin cells into 20 equal-occupancy bins of the quantile-normalised pseudotime, per species;
2. take the mean `log1p(CP10K)` expression per bin;
3. z-score each gene's profile *within* species, so the clustering sees shape and not amplitude;
4. concatenate → a 40-dimensional vector per gene pair (20 human bins ‖ 20 mouse bins).

Clustering that vector is what makes it *cross-species*: two genes land together only if they agree on the
shape in human **and** the shape in mouse.

**On choosing k.** The number of clusters is selected, not assumed — a sweep with two criteria that can
disagree: silhouette width (geometric separation) and bootstrap stability (does a refit on a gene subsample
reproduce the full-data labels). Both are reported for every k, and **whatever the criteria select is what
gets used.** The k = 4 solution is shown alongside regardless, because the up/up · up/down · down/up ·
down/down decomposition is the interpretable one — but if the criteria don't pick 4, this notebook says so
rather than forcing it.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score

NBIN, KRANGE, NBOOT = 20, list(range(2, 11)), 50
PROF_RUNS = ("pt_harmonized_mouse_nall", "pt_harmonized_human_nall")   # global-rank coordinate on both sides
have_prof = all(r in MI and (MID/r/"expr.npz").exists() for r in PROF_RUNS)
verify("prep matrices present for both full runs (needed for profiles)", have_prof,
       {r: (MID/r/"expr.npz").exists() for r in PROF_RUNS})

# conserved set: top decile of MI in BOTH full runs, over the 1:1 pairs
cons = compare(*PROF_RUNS)
p = PAIRS[PAIRS.human.isin(MI["pt_harmonized_human_nall"].index) & PAIRS.mouse.isin(MI["pt_harmonized_mouse_nall"].index)]
mm = MI["pt_harmonized_mouse_nall"].mi.reindex(pd.Index(p.mouse)).to_numpy()
hh = MI["pt_harmonized_human_nall"].mi.reindex(pd.Index(p.human)).to_numpy()
ok = np.isfinite(mm) & np.isfinite(hh); p = p[ok]; mm, hh = mm[ok], hh[ok]
k_top = int(round(TOPFRAC * len(p)))
sel = (pd.Series(mm).rank(ascending=False).to_numpy() <= k_top) & \
      (pd.Series(hh).rank(ascending=False).to_numpy() <= k_top)
CONS = p[sel].reset_index(drop=True)
print(f"universe {len(p):,} 1:1 pairs | top decile each side = {k_top:,} | conserved in both = {len(CONS):,}")
verify("conserved set matches the section-3 overlap count", len(CONS) == cons["overlap"],
       f"{len(CONS)} vs {cons['overlap']}")

In [ ]:
def binned_profiles(run, genes, nbin=NBIN):
    """Mean log1p expression in nbin equal-occupancy bins of pt_norm, z-scored per gene."""
    d = MID/run
    Xr = np.load(d/"expr.npz")["X"]                       # cells x genes, float32
    y  = np.load(d/"pseudotime.npy").astype(float)
    gn = pd.Index(np.load(d/"genes.npy", allow_pickle=True).astype(str))
    idx = gn.get_indexer(pd.Index(genes))
    assert (idx >= 0).all(), f"{run}: {(idx<0).sum()} requested genes absent from prep matrix"
    q = pd.qcut(pd.Series(y).rank(method="first"), nbin, labels=False)
    M = np.empty((len(genes), nbin), float)
    for b in range(nbin):
        M[:, b] = Xr[q.to_numpy() == b][:, idx].mean(axis=0)
    del Xr
    sd = M.std(axis=1, ddof=0); sd[sd == 0] = 1.0
    Z = (M - M.mean(axis=1, keepdims=True)) / sd[:, None]
    return M, Z, int(len(y))

Mm, Zm, n_m = binned_profiles("pt_harmonized_mouse_nall", CONS.mouse)
Mh, Zh, n_h = binned_profiles("pt_harmonized_human_nall", CONS.human)
print(f"profiles: mouse {Zm.shape} from {n_m:,} cells | human {Zh.shape} from {n_h:,} cells")
F = np.hstack([Zh, Zm])                                    # human bins || mouse bins
verify("feature matrix is finite", bool(np.isfinite(F).all()), F.shape)
verify("z-scored profiles have ~unit sd per species",
       bool(np.allclose(Zh.std(axis=1, ddof=0), 1, atol=1e-6) and
            np.allclose(Zm.std(axis=1, ddof=0), 1, atol=1e-6)))
# direction of each gene in each species: Spearman of its binned profile against bin index
from scipy.stats import spearmanr as _sr
bins = np.arange(NBIN)
rho_h = np.array([_sr(bins, Zh[i]).statistic for i in range(len(Zh))])
rho_m = np.array([_sr(bins, Zm[i]).statistic for i in range(len(Zm))])
print(f"direction: human up {int((rho_h>0).sum()):,} / down {int((rho_h<0).sum()):,} | "
      f"mouse up {int((rho_m>0).sum()):,} / down {int((rho_m<0).sum()):,}")

## 8.1 Selecting k — silhouette and bootstrap stability

In [ ]:
rows = []
LABELS = {}
for k in KRANGE:
    km = KMeans(n_clusters=k, n_init=20, random_state=SEED).fit(F)
    lab = km.labels_; LABELS[k] = lab
    sil = float(silhouette_score(F, lab)) if k > 1 else np.nan
    rng = np.random.default_rng(SEED); aris = []
    # SEEDED: with random_state=None the bootstrap ARI jittered ~0.01 between runs and the
    # argmax flipped between k=2 and k=3, so the notebook reported a different 'selected k'
    # on each execution. k=2/3/4 are genuinely indistinguishable (all ARI > 0.96) -- that is
    # the finding, but the reported selection must be deterministic.
    for rep in range(NBOOT):
        sub = rng.choice(len(F), size=int(0.8*len(F)), replace=False)
        km2 = KMeans(n_clusters=k, n_init=5, random_state=SEED*1000 + rep).fit(F[sub])
        aris.append(adjusted_rand_score(lab, km2.predict(F)))
    rows.append(dict(k=k, silhouette=round(sil, 4), boot_ari=round(float(np.mean(aris)), 4),
                     boot_ari_sd=round(float(np.std(aris)), 4),
                     min_cluster=int(pd.Series(lab).value_counts().min())))
    print(rows[-1], flush=True)
K = pd.DataFrame(rows)
K.to_csv(OUT/"traj_k_selection.csv", index=False)
k_sil  = int(K.loc[K.silhouette.idxmax(), "k"])
k_ari  = int(K.loc[K.boot_ari.idxmax(), "k"])
K_SEL  = k_sil if k_sil == k_ari else k_ari      # stability wins when the two disagree
print(f"\nbest silhouette k = {k_sil} | best bootstrap ARI k = {k_ari} | SELECTED k = {K_SEL}"
      f"{'  (criteria agree)' if k_sil == k_ari else '  (criteria DISAGREE -- stability preferred)'}")
verify("the two criteria agree on k", k_sil == k_ari, f"sil={k_sil}, ari={k_ari}",
       note="reported either way; selection uses stability when they differ")
# k=4 need not be the argmax. What matters for the quadrant reading is whether k=4 sits inside
# the plateau of good solutions -- i.e. is it within noise of the best, and does stability hold?
_best_sil, _best_ari = K.silhouette.max(), K.boot_ari.max()
_k4 = K[K.k == 4].iloc[0]
_sil_gap = float((_best_sil - _k4.silhouette) / _best_sil)
_ari_gap = float((_best_ari - _k4.boot_ari) / _best_ari)
print(f"k=4 vs the optimum: silhouette {_k4.silhouette:.4f} vs {_best_sil:.4f} ({100*_sil_gap:.1f}% lower)"
      f" | bootstrap ARI {_k4.boot_ari:.4f} vs {_best_ari:.4f} ({100*_ari_gap:.2f}% lower)")
_collapse = K[(K.k > 4) & (K.boot_ari < 0.92)].k.min()
print(f"stability collapses from k = {_collapse}" if np.isfinite(_collapse) else "no collapse in range")
verify("k=4 is within 10% of the best silhouette AND 5% of the best stability "
       "(so the quadrant reading is inside the plateau, not forced)",
       (_sil_gap < 0.10) and (_ari_gap < 0.05),
       f"sil {100*_sil_gap:.1f}% / ari {100*_ari_gap:.2f}% below optimum; argmax k={K_SEL}")

## 8.2 The k = 4 solution and the sign quadrants

The four *a priori* directional classes are up/up, up/down, down/up, down/down (human direction first). If the clustering is recovering that decomposition, each cluster should be dominated by one quadrant — that is the purity number below. Genes whose profile is essentially flat in one species (|ρ| near zero) have no meaningful sign, and the shape assignment is the more defensible call for those; they are counted separately.

In [ ]:
lab4 = LABELS[4]
quad = np.where(rho_h > 0, "up", "down")
quad = pd.Series([f"{a}/{b}" for a, b in zip(quad, np.where(rho_m > 0, "up", "down"))])
FLAT = 0.30
# rho_h/rho_m are numpy arrays, so `flat` is too -- do NOT call .to_numpy() on it
flat = np.asarray((np.abs(rho_h) < FLAT) | (np.abs(rho_m) < FLAT))
print(f"genes with |rho| < {FLAT} in at least one species (sign not meaningful): "
      f"{int(flat.sum()):,} of {len(quad):,} ({100*flat.mean():.1f}%)")

ct = pd.crosstab(pd.Series(lab4, name="cluster"), quad.rename("quadrant"))
print("\n=== cluster x quadrant, all conserved genes ===")
print(ct.to_string())
pur = (ct.max(axis=1) / ct.sum(axis=1)).round(3)
print("\nper-cluster purity (largest quadrant / cluster size):"); print(pur.to_string())
print(f"weighted mean purity: {float((ct.max(axis=1).sum())/ct.values.sum()):.3f}")

# INDEX ALIGNMENT BUG (fixed 2026-09-03): pd.Series(lab4[~flat]) gets a fresh 0..n-1 index while
# quad[~flat] keeps its ORIGINAL positional index, so pd.crosstab aligned on index and paired the
# wrong rows -- it returned purity 0.512, plausible-looking and wrong. Build both from numpy.
_q = quad.to_numpy()
ct2 = pd.crosstab(pd.Series(lab4[~flat], name="cluster"), pd.Series(_q[~flat], name="quadrant"))
assert int(ct2.values.sum()) == int((~flat).sum()), "crosstab lost rows to index alignment"
pur2 = (ct2.max(axis=1) / ct2.sum(axis=1)).round(3)
print(f"\n=== excluding the {int(flat.sum()):,} sign-ambiguous genes ===")
print(ct2.to_string()); print("\npurity:"); print(pur2.to_string())
print(f"weighted mean purity (unambiguous only): {float((ct2.max(axis=1).sum())/ct2.values.sum()):.3f}")
_p_all = float(ct.max(axis=1).sum()/ct.values.sum())
_p_un  = float(ct2.max(axis=1).sum()/ct2.values.sum())
verify("k=4 clusters map onto the sign quadrants at >=0.70 weighted purity", _p_un >= 0.70, round(_p_un, 4))
verify("dropping sign-ambiguous genes does not LOWER purity (their sign is arbitrary, so removing them "
       "can only help; a drop means index misalignment)",
       _p_un >= _p_all - 0.01, f"all {_p_all:.4f} -> unambiguous {_p_un:.4f}")
verify("all four quadrants are populated", ct.shape[1] == 4, list(ct.columns))

TRJ = CONS.copy()
TRJ["cluster_k4"] = lab4; TRJ["cluster_selected"] = LABELS[K_SEL]
TRJ["rho_human"] = rho_h.round(4); TRJ["rho_mouse"] = rho_m.round(4)
TRJ["quadrant"] = quad.to_numpy(); TRJ["sign_ambiguous"] = flat
TRJ["mi_human"] = MI["pt_harmonized_human_nall"].mi.reindex(pd.Index(CONS.human)).to_numpy().round(5)
TRJ["mi_mouse"] = MI["pt_harmonized_mouse_nall"].mi.reindex(pd.Index(CONS.mouse)).to_numpy().round(5)
TRJ.to_csv(OUT/"trajectory_clusters.csv", index=False)
print("\nwrote", OUT/"trajectory_clusters.csv")
for cl in sorted(set(lab4)):
    g = TRJ[TRJ.cluster_k4 == cl]
    print(f"  cluster {cl}: n={len(g):4d}  dominant {g.quadrant.mode().iloc[0]:9s} "
          f"({100*(g.quadrant==g.quadrant.mode().iloc[0]).mean():.0f}%)  e.g. "
          + ", ".join(g.nlargest(6, "mi_human").human.tolist()))

## 8.3 Figures — k selection, archetype profiles, quadrant map

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16.5, 4.3))
ax = axes[0]
ax.plot(K.k, K.silhouette, "o-", c="#4a7fb5", label="silhouette")
ax.set_xlabel("k"); ax.set_ylabel("silhouette width", color="#4a7fb5")
ax.tick_params(axis="y", labelcolor="#4a7fb5")
ax2 = ax.twinx()
ax2.errorbar(K.k, K.boot_ari, yerr=K.boot_ari_sd, fmt="s-", c="#c0392b", capsize=3, label="bootstrap ARI")
ax2.set_ylabel("bootstrap ARI (stability)", color="#c0392b"); ax2.tick_params(axis="y", labelcolor="#c0392b")
ax.axvline(K_SEL, c="k", ls="--", lw=1)
ax.text(K_SEL, ax.get_ylim()[1], f" selected k={K_SEL}", va="top", fontsize=9)
ax.set_title(f"k selection — silhouette peaks at {k_sil}, stability at {k_ari}", fontsize=10)

ax = axes[1]
ax.scatter(rho_h[~flat], rho_m[~flat], s=7, c=lab4[~flat],
           cmap="tab10", rasterized=True)
ax.scatter(rho_h[flat], rho_m[flat], s=5, facecolors="none",
           edgecolors="0.6", linewidths=0.4, rasterized=True, label=f"sign-ambiguous (n={int(flat.sum())})")
ax.axhline(0, c="k", lw=0.8); ax.axvline(0, c="k", lw=0.8)
for xx, yy, t in [(0.6,0.6,"up/up"), (-0.6,0.6,"down/up"), (0.6,-0.6,"up/down"), (-0.6,-0.6,"down/down")]:
    ax.text(xx, yy, t, ha="center", fontsize=9, alpha=0.75)
ax.set_xlabel("human: Spearman(profile, pseudotime bin)")
ax.set_ylabel("mouse: Spearman(profile, pseudotime bin)")
ax.set_title("Direction in each species, coloured by k=4 cluster", fontsize=10)
ax.legend(fontsize=7.5, loc="lower left", frameon=False)

ax = axes[2]
im = ax.imshow(ct.div(ct.sum(axis=1), axis=0).to_numpy(), cmap="Blues", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(ct.shape[1])); ax.set_xticklabels(ct.columns, rotation=45, ha="right", fontsize=9)
ax.set_yticks(range(ct.shape[0])); ax.set_yticklabels([f"cluster {i}" for i in ct.index], fontsize=9)
for i in range(ct.shape[0]):
    for j in range(ct.shape[1]):
        v = ct.iloc[i, j] / ct.iloc[i].sum()
        ax.text(j, i, f"{ct.iloc[i,j]}\n{100*v:.0f}%", ha="center", va="center", fontsize=8,
                c="white" if v > 0.55 else "k")
fig.colorbar(im, ax=ax, label="fraction of cluster")
ax.set_title("k=4 cluster vs sign quadrant", fontsize=10)
fig.tight_layout(); fig.savefig(OUT/"traj_k_and_quadrants.png", dpi=140, bbox_inches="tight")
print("wrote", OUT/"traj_k_and_quadrants.png")

In [ ]:
ks = sorted(set(lab4))
fig, axes = plt.subplots(2, len(ks), figsize=(3.5*len(ks), 6.4), sharex=True, sharey=True, squeeze=False)
xb = np.arange(NBIN)
for j, cl in enumerate(ks):
    m = lab4 == cl
    dom = TRJ[TRJ.cluster_k4 == cl].quadrant.mode().iloc[0]
    for i, (Z, spec, col) in enumerate(((Zh, "human", "#2c6fbb"), (Zm, "mouse", "#b5651d"))):
        ax = axes[i][j]
        ax.plot(xb, Z[m].T, c=col, alpha=0.06, lw=0.6, rasterized=True)
        mu, se = Z[m].mean(0), Z[m].std(0)/np.sqrt(m.sum())
        ax.plot(xb, mu, c=col, lw=2.4)
        ax.fill_between(xb, mu-1.96*se, mu+1.96*se, color=col, alpha=0.35, linewidth=0)
        ax.axhline(0, c="k", lw=0.6, ls=":")
        if i == 0: ax.set_title(f"cluster {cl}  n={int(m.sum())}\ndominant: {dom}", fontsize=10)
        if j == 0: ax.set_ylabel(f"{spec}\nz-scored expression")
        if i == 1: ax.set_xlabel("pseudotime bin (quantile)")
fig.suptitle("Conserved-MI gene trajectories, k=4 archetypes — mean ± 95% CI over genes, "
             "individual genes faint", y=1.01)
fig.tight_layout(); fig.savefig(OUT/"traj_archetypes_k4.png", dpi=140, bbox_inches="tight")
print("wrote", OUT/"traj_archetypes_k4.png")

if K_SEL != 4:
    labS = LABELS[K_SEL]; kss = sorted(set(labS))
    fig, axes = plt.subplots(2, len(kss), figsize=(3.3*len(kss), 6.2), sharex=True, sharey=True, squeeze=False)
    for j, cl in enumerate(kss):
        m = labS == cl
        for i, (Z, spec, col) in enumerate(((Zh, "human", "#2c6fbb"), (Zm, "mouse", "#b5651d"))):
            ax = axes[i][j]
            ax.plot(xb, Z[m].mean(0), c=col, lw=2.2); ax.axhline(0, c="k", lw=0.6, ls=":")
            if i == 0: ax.set_title(f"cluster {cl}  n={int(m.sum())}", fontsize=9)
            if j == 0: ax.set_ylabel(f"{spec}\nz-scored")
            if i == 1: ax.set_xlabel("pt bin")
    fig.suptitle(f"Selected k={K_SEL} archetypes (shown because the criteria did not pick 4)", y=1.01)
    fig.tight_layout(); fig.savefig(OUT/f"traj_archetypes_k{K_SEL}.png", dpi=140, bbox_inches="tight")
    print("wrote", OUT/f"traj_archetypes_k{K_SEL}.png")

json.dump(dict(k_selection=K.to_dict("records"), k_best_silhouette=k_sil, k_best_bootstrap_ari=k_ari,
               k_selected=K_SEL, n_conserved=int(len(CONS)), n_bins=NBIN, flat_threshold=FLAT,
               n_sign_ambiguous=int(flat.sum()),
               purity_k4_all=round(float(ct.max(axis=1).sum()/ct.values.sum()), 4),
               purity_k4_unambiguous=round(float(ct2.max(axis=1).sum()/ct2.values.sum()), 4),
               crosstab_k4=ct.to_dict(), profile_runs=list(PROF_RUNS)),
          open(OUT/"trajectory_clusters_report.json","w"), indent=1, default=str)
bad = [l for l, o, _, _ in CHECKS if not o]
print(f"\nFINAL: {len(CHECKS)} checks | {len(bad)} FAILED" + (": " + "; ".join(bad) if bad else " — all passed"))

---
# 9. Specificity on a SHARED universe, with fixed top-N

Section 3 gave each comparison its own intersection universe. That is the permissive version and it stays
in the table — but fold enrichment is `observed / (k²/N)`, so it is **N-dependent**, and the whole point of
the specificity comparison is putting the uninduced bar next to the induced bar. Different N makes those
two bars not strictly comparable.

Here every arm is restricted to **one shared universe** — the 1:1 pairs measurable in *all* runs — so N and
k are identical across bars. And the cutoff is taken as a **fixed top-N** (100, 200, 500, 1000, 2000) as
well as a fraction, so the result does not depend on how much of each species' noise tail a decile happens
to scoop up.

In [ ]:
MOUSE_ARMS = [r for r in ("pt_harmonized_unind_mouse_nall","pt_harmonized_indmatch_mouse_nall","pt_harmonized_mouse_nall") if r in MI]
HUM = "pt_harmonized_human_nall"
have_shared = bool(MOUSE_ARMS) and HUM in MI

if have_shared:
    U = PAIRS.copy()
    U = U[U.human.isin(MI[HUM].index)]
    for r in MOUSE_ARMS: U = U[U.mouse.isin(MI[r].index)]
    hv = MI[HUM].mi.reindex(pd.Index(U.human)).to_numpy()
    mv = {r: MI[r].mi.reindex(pd.Index(U.mouse)).to_numpy() for r in MOUSE_ARMS}
    ok = np.isfinite(hv)
    for r in MOUSE_ARMS: ok &= np.isfinite(mv[r])
    U = U[ok].reset_index(drop=True); hv = hv[ok]; mv = {r: v[ok] for r, v in mv.items()}
    NU = len(U)
    print(f"SHARED universe: {NU:,} 1:1 pairs measurable in {HUM} and all of {MOUSE_ARMS}")
    verify("shared universe is a subset of every per-pair universe",
           bool(NU <= R.universe.min()) if len(R) else None, f"{NU:,} vs min per-pair {R.universe.min():,}")

    def shared_cmp(arm, k):
        th = set(np.argsort(-hv)[:k]); ta = set(np.argsort(-mv[arm])[:k])
        ov = len(th & ta); exp = k*k/NU
        return dict(arm=arm, k=k, N=NU, overlap=ov, expected=round(exp,1),
                    fold=round(ov/exp,3) if exp else np.nan,
                    p=float(hypergeom.sf(ov-1, NU, k, k)),
                    rho=round(float(spearmanr(hv, mv[arm]).statistic), 4))

    KS_FIXED = [k for k in (100, 200, 500, 1000, 2000) if k < NU//2]
    KS_FRAC  = [int(round(f*NU)) for f in (0.02, 0.05, 0.10, 0.20)]
    SH = pd.DataFrame([shared_cmp(a, k) for k in sorted(set(KS_FIXED+KS_FRAC)) for a in MOUSE_ARMS])
    SH["cutoff"] = np.where(SH.k.isin(KS_FIXED), "top-" + SH.k.astype(str),
                            (100*SH.k/NU).round(0).astype(int).astype(str) + "%")
    print("\n=== fold enrichment vs human, shared universe ===")
    print(SH.pivot_table(index="k", columns="arm", values="fold").round(3).to_string())
    SH.to_csv(OUT/"specificity_shared_universe.csv", index=False)

In [ ]:
if have_shared and {"pt_harmonized_unind_mouse_nall","pt_harmonized_indmatch_mouse_nall"} <= set(MOUSE_ARMS):
    piv = SH.pivot_table(index="k", columns="arm", values="fold")
    piv["ratio_unind_over_induced"] = (piv.pt_harmonized_unind_mouse_nall / piv.pt_harmonized_indmatch_mouse_nall).round(3)
    print("=== the number that carries the argument ===")
    print(piv[["pt_harmonized_unind_mouse_nall","pt_harmonized_indmatch_mouse_nall","ratio_unind_over_induced"]].round(3).to_string())
    ratios = piv.ratio_unind_over_induced.dropna()
    print(f"\nratio across all cutoffs: min {ratios.min():.3f}  median {ratios.median():.3f}  max {ratios.max():.3f}")
    verify("uninduced enrichment is within 2x of induced at EVERY cutoff (not decidualization-specific)",
           bool(((ratios > 0.5) & (ratios < 2.0)).all()), dict(ratios.round(3)))
    verify("uninduced is enriched above chance at every cutoff",
           bool((piv.pt_harmonized_unind_mouse_nall > 1.0).all()), dict(piv.pt_harmonized_unind_mouse_nall.round(3)))
    # composition of the uninduced arm -- why this is a strong version of the claim
    bm = META["pt_harmonized_unind_mouse_nall"]["build"]
    print(f"\nuninduced arm: {bm.get('n_cells'):,} cells, normalization {bm.get('normalization')}")

## 9.1 The presentation panel

One figure and one sentence, written to standalone files so they can go straight into a slide. The claim is about *which genes* the axis selects, not about the pseudotime values — the two arms are ranked separately because raw uninduced dpt spans only a fraction of the global axis.

In [ ]:
if have_shared and {"pt_harmonized_unind_mouse_nall","pt_harmonized_indmatch_mouse_nall"} <= set(MOUSE_ARMS):
    NICE = {"pt_harmonized_unind_mouse_nall":"uninduced\n(resting uterus)", "pt_harmonized_indmatch_mouse_nall":"induced\n(size-matched)",
            "pt_harmonized_mouse_nall":"full\ntimecourse"}
    COL  = {"pt_harmonized_unind_mouse_nall":"#3d7ea6", "pt_harmonized_indmatch_mouse_nall":"#c0392b", "pt_harmonized_mouse_nall":"0.55"}
    fig, axes = plt.subplots(1, 2, figsize=(12.6, 4.9))
    ax = axes[0]
    ks_plot = sorted(set(KS_FIXED))
    w = 0.8/len(MOUSE_ARMS)
    for i, arm in enumerate(MOUSE_ARMS):
        g = SH[(SH.arm == arm) & (SH.k.isin(ks_plot))].sort_values("k")
        ax.bar(np.arange(len(g)) + i*w, g.fold, width=w, color=COL[arm], edgecolor="k",
               linewidth=0.5, label=NICE[arm].replace("\n"," "))
    ax.axhline(1.0, c="k", ls="--", lw=1.2)
    ax.text(-0.35, 1.03, "chance", fontsize=8)
    ax.set_xticks(np.arange(len(ks_plot)) + w*(len(MOUSE_ARMS)-1)/2)
    ax.set_xticklabels([f"top {k}" for k in ks_plot], fontsize=9)
    ax.set_ylabel("fold enrichment of shared\ntop-N pseudotime-MI genes vs human")
    ax.set_xlabel(f"cutoff (shared universe, N = {NU:,} 1:1 orthologs)")
    ax.legend(fontsize=8.5, frameon=False, title="mouse arm", title_fontsize=8.5)
    ax.set_title("Conservation is present at full strength in resting uterus", fontsize=11)

    ax = axes[1]
    for arm in MOUSE_ARMS:
        g = SH[SH.arm == arm].sort_values("k")
        ax.plot(g.k, g.fold, "o-", c=COL[arm], lw=1.8, ms=4.5, label=NICE[arm].replace("\n"," "))
    ax.axhline(1.0, c="k", ls="--", lw=1.2); ax.set_xscale("log")
    ax.set_xlabel("top-N cutoff (log scale)"); ax.set_ylabel("fold enrichment")
    ax.legend(fontsize=8.5, frameon=False)
    ax.set_title("Stable across every cutoff — no threshold to argue about", fontsize=11)
    fig.text(0.5, -0.045, "Uninduced = 2 sections from 1 animal (pseudo-replicates); the induced arm is "
             "replicated. Arms ranked separately: raw uninduced dpt spans 0–0.425 of the global axis.",
             ha="center", fontsize=8, style="italic")
    fig.tight_layout(); fig.savefig(OUT/"specificity_slide_pt.png", dpi=200, bbox_inches="tight")
    print("wrote", OUT/"specificity_slide_pt.png")

    r10 = piv.ratio_unind_over_induced.dropna()
    txt = f"""GENES CORRELATED WITH PSEUDOTIME ARE NOT DECIDUALIZATION-SPECIFIC
{'='*72}

Genes carrying information about the pseudotime axis are conserved between human
and mouse -- but the same conservation is present at full strength in UNINDUCED
uterus, which is not decidualizing. The pseudotime-informative program is therefore
generic stromal/uterine, not decidualization-specific.

Shared universe: {NU:,} strict 1:1 Ensembl-116 orthologs measurable in every arm.
Fold enrichment vs human, by fixed top-N cutoff:

{piv[[c for c in ('pt_harmonized_unind_mouse_nall','pt_harmonized_indmatch_mouse_nall','pt_harmonized_mouse_nall') if c in piv]].round(3).to_string()}

Uninduced / induced ratio: min {r10.min():.3f}, median {r10.median():.3f}, max {r10.max():.3f}
Threshold-free Spearman of MI ranks vs human:
{SH.drop_duplicates('arm').set_index('arm').rho.round(4).to_string()}

WHY THIS IS THE STRONG VERSION
  The uninduced arm is ~97% fibroblast (9,270 of 9,334 cells are efib_*; only 64
  decidual). Its pseudotime is variation among RESTING fibroblasts, not a
  decidualization trajectory -- so a coordinate with no decidualization in it still
  selects the genes human's full trajectory selects.

CAVEATS TO STATE
  - Uninduced = 2 sections from 1 animal (pseudo-replicates). The induced arm is
    properly replicated, so the comparison is uninduced-vs-a-distribution.
  - Arms are rank-normalised separately; the claim is about WHICH GENES the axis
    selects, not about the pseudotime values.
  - pt_harmonized_mouse_nall / pt_harmonized_human_nall are pooled (global-rank) estimates, not batch-conditioned.

THE COUNTERPOINT SLIDE (from the isodepth analysis, already computed)
  37 genes ACQUIRE conserved positional information only upon decidualization
  (incl. RUNX1, CEBPB, EBF1); 31 LOSE theirs. So: the axis and most of its gene
  repertoire are generic uterine stroma -- and this is the small specific set that
  decidualization actually adds.
"""
    (OUT/"specificity_slide_pt.txt").write_text(txt)
    print(txt)

---
# 10. Regulators on the integrated pseudotime

The equivalent of the isodepth regulator analysis, on the integrated (global-rank) pseudotime. **Expression
only** — SCENIC contributes gene *names* so `is_tf` can be an annotation column; no regulon activity, no
AUCell, no GRN edge enters any statistic here.

Verdicts are named `verdict_pt` deliberately. The isodepth verdicts carried a per-sample **consistency**
requirement (top-percentile *and* consistent in ≥⅔ of sections in both species); these pooled runs have no
per-sample decomposition, so `verdict_pt` rests on percentile plus direction and shape agreement only. Same
name would have implied the same bar.

In [ ]:
# SCENIC contributes NAMES ONLY -- is_tf is an annotation column and never enters a statistic.
# Plain line reading: pd.read_csv(sep=None, engine="python") raises on a single-column list of
# gene symbols (delimiter sniffing has nothing to sniff), which is why an earlier run reported
# the list as absent when it was sitting right there.
TF_CANDIDATES = [
    "/n/lab_storage/eddy_lab/users/nhilgert/decidualization/revisions/scenic/databases/allTFs_hg38.txt",
    "/n/lab_storage/eddy_lab/users/nhilgert/decidualization/revisions/scenic/databases/allTFs_mm.txt",
]
TFS, tf_src = set(), None
for f in TF_CANDIDATES:
    p = pathlib.Path(f)
    if not p.exists(): continue
    names = {ln.strip().split()[0] for ln in p.read_text().splitlines() if ln.strip()}
    if 500 < len(names) < 5000:
        TFS, tf_src = names, f
        break
print(f"TF list: {len(TFS):,} names from {tf_src}" if TFS else
      "TF list NOT FOUND -- is_tf all-False, TF contrasts skipped")
verify("TF list located (annotation only, never enters a statistic)", bool(TFS),
       f"{len(TFS):,} from {pathlib.Path(tf_src).name}" if tf_src else None)
verify("TF list overlaps the human gene symbols in the ranking universe",
       bool(TFS) and len(TFS & set(PAIRS.human)) > 200,
       len(TFS & set(PAIRS.human)) if TFS else None)

In [ ]:
if have_prof:
    # conservation ranking over the FULL shared 1:1 universe, not just the conserved subset
    Uf = PAIRS[PAIRS.human.isin(MI[HUM].index) & PAIRS.mouse.isin(MI["pt_harmonized_mouse_nall"].index)]
    h_mi = MI[HUM].mi.reindex(pd.Index(Uf.human)).to_numpy()
    m_mi = MI["pt_harmonized_mouse_nall"].mi.reindex(pd.Index(Uf.mouse)).to_numpy()
    ok = np.isfinite(h_mi) & np.isfinite(m_mi)
    Uf = Uf[ok].reset_index(drop=True); h_mi, m_mi = h_mi[ok], m_mi[ok]
    G = pd.DataFrame(dict(human=Uf.human.to_numpy(), mouse=Uf.mouse.to_numpy(),
                          mi_human=h_mi.round(5), mi_mouse=m_mi.round(5)))
    G["pct_human"] = pd.Series(h_mi).rank(pct=True).round(4)
    G["pct_mouse"] = pd.Series(m_mi).rank(pct=True).round(4)
    G["min_pct"]   = G[["pct_human","pct_mouse"]].min(axis=1)
    G["is_tf"]     = G.human.isin(TFS)
    print(f"ranking universe: {len(G):,} 1:1 pairs | TFs among them: {int(G.is_tf.sum()):,}")

    # Direction + shape need binned profiles, which are expensive, so only a slice gets them.
    # BUG FIXED 2026-09-03: the slice was the top N by MIN_pct, which by construction excludes the
    # species-specific class (high in one species, low in the other -> low min_pct). That would have
    # reported a species_specific count near zero as though it were a finding. Select on MAX_pct
    # instead -- genes informative in AT LEAST ONE species -- which covers transferable, inverted and
    # species-specific alike.
    G["max_pct"] = G[["pct_human","pct_mouse"]].max(axis=1)
    TOPN_PROF = min(3000, len(G))
    Gt = G.nlargest(TOPN_PROF, "max_pct").reset_index(drop=True)
    print(f"profile slice: top {TOPN_PROF:,} by max_pct | min_pct range {Gt.min_pct.min():.3f}"
          f"-{Gt.min_pct.max():.3f} (spans both conserved and species-specific)")
    _, Zh2, _ = binned_profiles("pt_harmonized_human_nall", Gt.human)
    _, Zm2, _ = binned_profiles("pt_harmonized_mouse_nall", Gt.mouse)
    bins = np.arange(NBIN)
    Gt["rho_human"] = [round(float(_sr(bins, Zh2[i]).statistic), 4) for i in range(len(Gt))]
    Gt["rho_mouse"] = [round(float(_sr(bins, Zm2[i]).statistic), 4) for i in range(len(Gt))]
    Gt["profile_r"] = [round(float(np.corrcoef(Zh2[i], Zm2[i])[0,1]), 4) for i in range(len(Gt))]
    Gt["peak_human"] = Zh2.argmax(axis=1); Gt["peak_mouse"] = Zm2.argmax(axis=1)
    Gt["dpeak"] = Gt.peak_human - Gt.peak_mouse
    Gt["sign_concordant"] = np.sign(Gt.rho_human) == np.sign(Gt.rho_mouse)

    HI = 0.90
    Gt["verdict_pt"] = np.where(
        (Gt.pct_human >= HI) & (Gt.pct_mouse >= HI) & Gt.sign_concordant & (Gt.profile_r > 0.5), "transferable",
        np.where((Gt.pct_human >= HI) & (Gt.pct_mouse >= HI) & (~Gt.sign_concordant), "inverted",
        np.where((Gt.pct_human >= HI) ^ (Gt.pct_mouse >= HI), "species_specific", "unresolved")))
    print("\n=== verdict_pt (percentile + direction + shape; NO consistency requirement) ===")
    print(Gt.verdict_pt.value_counts().to_string())
    print("\nsign concordance:", f"{100*Gt.sign_concordant.mean():.1f}%",
          "| median profile_r:", round(float(Gt.profile_r.median()), 4))
    print("dpeak: median", int(Gt.dpeak.median()), "| IQR",
          f"{int(Gt.dpeak.quantile(.25))}..{int(Gt.dpeak.quantile(.75))}",
          "| frac |dpeak|>=5 bins:", round(float((Gt.dpeak.abs()>=5).mean()), 3))
    if TFS.__len__():
        print("\nTF vs non-TF, among the top slice:")
        print(Gt.groupby("is_tf").agg(n=("human","size"), median_min_pct=("min_pct","median"),
                                      median_profile_r=("profile_r","median"),
                                      frac_transferable=("verdict_pt", lambda s: round((s=="transferable").mean(),3)),
                                      median_abs_dpeak=("dpeak", lambda s: float(s.abs().median()))).to_string())
    Gt.to_csv(OUT/"regulators_pt.csv", index=False)
    G.to_csv(OUT/"conservation_ranking_pt_all_pairs.csv", index=False)
    verify("verdict_pt populates all four classes", Gt.verdict_pt.nunique() == 4,
           dict(Gt.verdict_pt.value_counts()))
    verify("species_specific class is non-trivial (would be ~0 under the old min_pct pre-filter)",
           int((Gt.verdict_pt == "species_specific").sum()) > 50,
           int((Gt.verdict_pt == "species_specific").sum()))
    verify("the profile slice spans low min_pct (i.e. includes one-sided genes)",
           float(Gt.min_pct.min()) < 0.5, round(float(Gt.min_pct.min()), 3))

## 10.1 Pseudotime vs isodepth — does the coordinate change the answer?

In [ ]:
ISO = None
for cand in (X/"notebooks/verify/data/conservation_ranking_all_genes.tsv",
             X/"results/tables/conservation_ranking_all_genes.tsv"):
    if cand.exists(): ISO = pd.read_csv(cand, sep="\t"); print("isodepth ranking:", cand); break
if ISO is None:
    print("isodepth conservation ranking NOT FOUND -- section 10.1 skipped")
else:
    key = "human" if "human" in ISO.columns else ISO.columns[0]
    I = ISO.set_index(key)
    common = G.set_index("human").index.intersection(I.index)
    print(f"genes in both rankings: {len(common):,}")
    a = G.set_index("human").loc[common, "min_pct"]
    bcol = "min_pct" if "min_pct" in I.columns else [c for c in I.columns if "min_pct" in c or "pct" in c][0]
    b = pd.to_numeric(I.loc[common, bcol], errors="coerce")
    m = a.notna() & b.notna()
    print(f"Spearman(min_pct on pseudotime, min_pct on isodepth) = {spearmanr(a[m], b[m]).statistic:+.4f}")
    for k in (100, 200, 500, 1000):
        ta = set(a[m].nlargest(k).index); tb = set(b[m].nlargest(k).index)
        exp = k*k/int(m.sum())
        print(f"  top {k:5d}: overlap {len(ta&tb):5d}  expected {exp:7.1f}  fold {len(ta&tb)/exp:6.2f}")
    if TFS:
        tf_both = [g for g in common if g in TFS]
        print(f"\nTFs in both rankings: {len(tf_both):,}")
        cmp_tf = pd.DataFrame(dict(pt=a.reindex(tf_both), iso=b.reindex(tf_both))).dropna()
        cmp_tf["delta"] = (cmp_tf.pt - cmp_tf.iso).round(4)
        print("\nTFs that gain most on pseudotime:"); print(cmp_tf.nlargest(12,"delta").round(4).to_string())
        print("\nTFs that lose most on pseudotime:"); print(cmp_tf.nsmallest(12,"delta").round(4).to_string())
        for g in ("RUNX1","CEBPB","EBF1","FOXO1","MEIS2","NFIA","RORA","WT1"):
            if g in cmp_tf.index:
                r = cmp_tf.loc[g]
                print(f"  {g:7s} pseudotime pct {r.pt:.3f} | isodepth pct {r.iso:.3f} | delta {r.delta:+.3f}")
    cmp_tf.to_csv(OUT/"tf_pt_vs_isodepth.csv") if TFS else None

In [ ]:
bad = [l for l, o, _, _ in CHECKS if not o]
print(f"NOTEBOOK COMPLETE: {len(CHECKS)} checks | {len(bad)} FAILED"
      + (":\n  - " + "\n  - ".join(bad) if bad else " — all passed"))
print("\nall outputs:", OUT)
for f in sorted(OUT.glob("*")): print("   ", f.name)